# 21 — Data Visualization

**ENAI 603 Capstone — WMATA Metro Delay Prediction**

This notebook provides map visualizations of WMATA rail stations and line predictions from the data collected.

**Sections:**
1. Load WMATA Stations dataset
2. Load WMATA Predictions from features.csv file
3. Map WMATA Rail System network
4. Add HeatMap layer weighted by number of trains at station
5. Add HeatMap layer weighted by delays at station
6. Map WMATA Red line
7. Map WMATA Blue line
8. Map WMATA Green line
9. Map WMATA Yellow line
10. Map WMATA Orange line
11. Map WMATA Silver line

## 1. Load WMATA Stations dataset

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd
import numpy as np

import folium
import folium.plugins as plugins
from folium.plugins import OverlappingMarkerSpiderfier, HeatMap

from fuzzywuzzy import process
import copy

In [ ]:
# Paths
PROJECT_DIR = Path("..")
DB_PATH = PROJECT_DIR / "data" / "wmata.db"

# Load WMATA database
conn = sqlite3.connect(DB_PATH)
print(f"Connected to {DB_PATH}")
print(f"Database size: {DB_PATH.stat().st_size / 1024:.0f} KB")

In [ ]:
# Load stations dataset
df_stations = pd.read_sql("SELECT * FROM stations", conn)
print(f"Total stations: {len(df_stations)}")
df_stations.head(5)

## 2. Load WMATA predictions from features.csv file

In [ ]:
# Load predictions from features.csv file
df_pred = pd.read_csv(PROJECT_DIR / "data" / "features.csv")
df_pred.head(5)

---

## 3. Map WMATA Rail System network

In [ ]:
# Route sequences for each line retrieved from https://www.wmata.com/schedules/maps/upload/system-map-rail.pdf
RAIL_MAP = {
  "RD": ['shady', 'rockville', 'twinbrook', 'north bethesda', 'grosvenor', 'medical', 'bethesda', 'friendship', 'tenleytown', 'van ness', 'cleveland', 'woodley', 'dupont', 'farragut north', 'metro',
         'gallery', 'judiciary', 'union', 'noma', 'rhode', 'brookland', 'fort', 'takoma', 'silver', 'forest', 'wheaton', 'glenmont'],
  "BL": ['franconia', 'van dorn', 'king', 'braddock', 'potomac yard', 'reagan', 'crystal', 'pentagon city', 'pentagon', 'arlington', 'rosslyn', 'foggy', 'farragut west', 'mcpherson', 'metro',
         'federal triangle', 'smithsonian', 'enfant', 'federal center', 'capitol south', 'eastern', 'potomac ave', 'stadium', 'benning', 'capitol heights', 'addison', 'morgan boulevard', 'largo'],
  "GR": ['branch', 'suitland', 'naylor', 'southern', 'congress', 'anacostia', 'ballpark', 'waterfront', 'enfant', 'archives', 'gallery',
           'vernon', 'shaw', 'cardozo', 'columbia', 'georgia', 'fort', 'west hyattsville', 'hyattsville crossing', 'college', 'greenbelt'],
  "YL": [['huntington', 'eisenhower', 'king', 'braddock', 'potomac yard', 'reagan', 'crystal', 'pentagon city', 'pentagon', 'enfant', 'archives', 'gallery',
         'vernon', 'shaw', 'cardozo', 'columbia', 'georgia', 'fort', 'west hyattsville', 'hyattsville crossing', 'college', 'greenbelt'],
         ['huntington', 'eisenhower', 'king', 'braddock', 'potomac yard', 'reagan', 'crystal', 'pentagon city', 'pentagon', 'enfant', 'archives', 'gallery',
         'vernon']],
  "OR": ['vienna', 'dunn', 'west falls', 'east falls', 'ballston', 'virginia', 'clarendon', 'court', 'rosslyn', 'foggy', 'farragut west', 'mcpherson', 'metro',
         'federal triangle', 'smithsonian', 'enfant', 'federal center', 'capitol south', 'eastern', 'potomac ave', 'stadium', 'minnesota', 'deanwood', 'cheverly', 'landover', 'carrollton'],
  "SV": [['ashburn', 'loudoun', 'dulles', 'innovation', 'herndon', 'reston town', 'wiehle', 'spring hill', 'greensboro', 'tysons', 'mclean', 'east falls', 'ballston', 'virginia', 'clarendon', 'court', 'rosslyn', 'foggy', 'farragut west', 'mcpherson', 'metro',
         'federal triangle', 'smithsonian', 'enfant', 'federal center', 'capitol south', 'eastern', 'potomac ave', 'stadium', 'minnesota', 'deanwood', 'cheverly', 'landover', 'carrollton'],
         ['ashburn', 'loudoun', 'dulles', 'innovation', 'herndon', 'reston town', 'wiehle', 'spring hill', 'greensboro', 'tysons', 'mclean', 'east falls', 'ballston', 'virginia', 'clarendon', 'court', 'rosslyn', 'foggy', 'farragut west', 'mcpherson', 'metro',
         'federal triangle', 'smithsonian', 'enfant', 'federal center', 'capitol south', 'eastern', 'potomac ave', 'stadium', 'benning', 'capitol heights', 'addison', 'morgan boulevard', 'largo']]}

# Flatten an array (or tuple)
def flatten(arr):
  flat_list = []
  for row in arr:
    if isinstance(row, str): return arr
    flat_list.extend(row)
      
  return flat_list

# Extract unique values from an array (or tuple)
def unique(arr):
  return list(dict.fromkeys(arr))

In [ ]:
# Extract Metro Center station coordinates from WMATA Stations dataset
lat, lon = df_stations.loc[df_stations['station_name'] == 'Metro Center', ['lat', 'lon']].iloc[0]

# Create a Folium map centered on Metro Center station
m = folium.Map(
  location = [lat, lon],
  zoom_start = 11
)

# Add Markers for each station with popup info
station_names = list(df_stations['station_name'])
line_colors = {"RD": "red", "BL": "blue", "OR": "orange", "SV": "silver",
               "GR": "lightgreen", "YL": "yellow"}

for line, stations in RAIL_MAP.items():
  stations = unique(flatten(stations))  # Handle lines with +2 terminals
    
  for stop in stations:
    best_match = process.extractOne(stop, station_names)
    station = df_stations[df_stations['station_name'] == best_match[0]].iloc[0]

    popup_html = f"""
    <b>Station {station['station_code']}:</b><br>
    [{line}] {station['station_name']}
    """

    color = line_colors[line]
    folium.Marker(
        location=[station['lat'], station['lon']],
        popup=folium.Popup(popup_html, max_width=200),
        icon=plugins.BeautifyIcon(
                   icon="train",
                   icon_shape="circle",
                   background_color=color
               ),
        tooltip=f"<b>Click to see lines at</b><br>Station {station['station_code']}: {station['station_name']}"
    ).add_to(m)
      
# Manage lines that share station stops (overlapping markers) by "spiderfying" them when clicked
oms = OverlappingMarkerSpiderfier(
    keep_spiderfied=True,  # Markers remain spiderfied after clicking
    nearby_distance=20,  # Distance for clustering markers in pixel
    circle_spiral_switchover=10,  # Threshold for switching between circle and spiral
    leg_weight=2.0  # Line thickness for spider legs
    )
oms.add_to(m)

m

## 4. Add HeatMap layer weighted by number of trains at station

In [ ]:
# Add congestion HeatMap layer weighted by number of trains at station
m_copy = copy.deepcopy(m)

HeatMap(df_pred[['lat', 'lon', 'num_trains_at_station']]).add_to(m_copy)
m_copy

## 5. Add HeatMap layer weighted by delays at station

In [ ]:
# Add HeatMap layer weighted by delays at station
HeatMap(df_pred[['lat', 'lon', 'is_delayed']]).add_to(m)
m

## 6. Map WMATA Red line

In [ ]:
def add_stations_markers(m, line_code, line_stations):
  line_stations = unique(flatten(line_stations))

  for stop in line_stations:
    best_match = process.extractOne(stop, station_names)
    station = df_stations[df_stations['station_name'] == best_match[0]].iloc[0]

    color = line_colors[line_code]
    folium.Marker(
        location=[station['lat'], station['lon']],
        icon=plugins.BeautifyIcon(
                   icon="train",
                   icon_shape="circle",
                   background_color=color
               ),
        tooltip=f"<b>Station {station['station_code']}: {station['station_name']}"
    ).add_to(m)

In [ ]:
# Red line map
m1 = folium.Map(
  location = [lat, lon],
  zoom_start = 11
)

add_stations_markers(m1, 'RD', RAIL_MAP['RD'])

m1

## 7. Map WMATA Blue line

In [ ]:
# Blue line map
m2 = folium.Map(
  location = [lat, lon],
  zoom_start = 11
)

add_stations_markers(m2, 'BL', RAIL_MAP['BL'])

m2

## 8. Map WMATA Green line

In [ ]:
# Green line map
m3 = folium.Map(
  location = [lat, lon],
  zoom_start = 11
)

add_stations_markers(m3, 'GR', RAIL_MAP['GR'])

m3

## 9. Map WMATA Yellow line

In [ ]:
# Yellow line map
m4 = folium.Map(
  location = [lat, lon],
  zoom_start = 11
)

add_stations_markers(m4, 'YL', RAIL_MAP['YL'])

m4

## 10. Map WMATA Orange line

In [ ]:
# Orange line map
m5 = folium.Map(
  location = [lat, lon],
  zoom_start = 11
)

add_stations_markers(m5, 'OR', RAIL_MAP['OR'])

m5

## 11. Map WMATA Silver line

In [ ]:
# Silver line map
m6 = folium.Map(
  location = [lat, lon],
  zoom_start = 11
)

add_stations_markers(m6, 'SV', RAIL_MAP['SV'])

m6